# EnergyLens-AI - Notebook 6: Final Pipeline

This last notebook ties everything together into one pipeline.

What's in here:
1. Train the household-level XGBoost forecaster on the full feature matrix.
2. Train the scaler and KMeans clusterer on the household profiles from Notebook 5.
3. Save all the trained models with joblib so they can be reused without retraining.
4. A simple recommendation function that gives suggestions based on which cluster a household belongs to.
5. An `EnergyLensPipeline` class that, given a household ID, recent history and a weather forecast, returns a 7-day forecast, flags any anomalies, and gives personalized recommendations.

Note: Notebook 4 compared forecasting models at the system-wide (aggregate) level, but here I train a per-household XGBoost model instead, since personalization, clustering and recommendations all need to operate at the household level.

I didn't want this to just stop at "here's the best model" - a class that loads the saved models, builds the features, runs the forecast, checks for anomalies and returns recommendations is closer to something that could actually be plugged into an app or API.


In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os
import gc
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib

# Check if running in Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"✅ Setup complete. Running in Colab: {IN_COLAB}")


✅ Setup complete. Running in Colab: True


## 2. Load Data

Loading the feature matrix from Notebook 3 and the household profiles (with cluster labels) from Notebook 5.


In [2]:
if IN_COLAB:
    from google.colab import files
    print('📤 Please upload master_features.parquet')
    uploaded_feats = files.upload()
    features_path = list(uploaded_feats.keys())[0]

    print('\n📤 Please upload household_profiles.csv')
    uploaded_profiles = files.upload()
    profiles_path = list(uploaded_profiles.keys())[0]
else:
    features_path = '../data/processed/master_features.parquet'
    profiles_path = '../data/processed/household_profiles.csv'

df = pd.read_parquet(features_path)
profiles = pd.read_csv(profiles_path)
df['day'] = pd.to_datetime(df['day'])

print(f'✅ Features loaded: {df.shape}')
print(f'✅ Profiles loaded: {profiles.shape}')

📤 Please upload master_features.parquet


Saving master_features.parquet to master_features.parquet

📤 Please upload household_profiles.csv


Saving household_profiles.csv to household_profiles.csv
✅ Features loaded: (3332960, 58)
✅ Profiles loaded: (5530, 6)


## 3. Training the Final Models

Training on all available data this time (not a train/test split), since these are the models we're actually going to save and use for inference.


In [3]:
# Create models directory
models_dir = '/content/models' if IN_COLAB else '../models'
os.makedirs(models_dir, exist_ok=True)

# Train and save the XGBoost forecaster
# Set features and target
target = 'energy_mean'
features = [
    'energy_lag_1', 'energy_lag_7', 'energy_lag_14', 'energy_lag_28',
    'energy_roll_mean_7', 'energy_roll_std_7', 'energy_roll_mean_30',
    'temp_avg', 'HDD', 'CDD', 'temp_range', 'is_weekend', 'is_holiday',
    'tariff_code', 'acorn_code'
]

print("Training final XGBoost Forecaster...")
X = df[features]
y = df[target]

xgb_forecaster = xgb.XGBRegressor(n_estimators=120, learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1)
xgb_forecaster.fit(X, y)

forecaster_path = os.path.join(models_dir, 'xgboost_forecaster.joblib')
joblib.dump(xgb_forecaster, forecaster_path)
print(f"✅ Saved forecaster to: {forecaster_path}")


Training final XGBoost Forecaster...
✅ Saved forecaster to: /content/models/xgboost_forecaster.joblib


In [4]:
# Train and save the scaler and KMeans clusterer
cluster_features = ['mean_daily_consumption', 'std_consumption', 'thermal_sensitivity', 'weekend_bias']

print("Training Scaler & KMeans Clusterer...")
scaler = StandardScaler()
scaled_X = scaler.fit_transform(profiles[cluster_features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(scaled_X)

# Save binaries
scaler_path = os.path.join(models_dir, 'scaler.joblib')
clusterer_path = os.path.join(models_dir, 'kmeans_clusterer.joblib')

joblib.dump(scaler, scaler_path)
joblib.dump(kmeans, clusterer_path)

print(f"✅ Saved scaler to: {scaler_path}")
print(f"✅ Saved clusterer to: {clusterer_path}")


Training Scaler & KMeans Clusterer...
✅ Saved scaler to: /content/models/scaler.joblib
✅ Saved clusterer to: /content/models/kmeans_clusterer.joblib


## 4. Recommendation Rules

Simple rules based on which cluster a household falls into:
- Cluster 0 (thermal/heating sensitive) - insulation and smart thermostat suggestions
- Cluster 1 (frugal/efficient) - time-of-use tariff and off-peak scheduling suggestions
- Cluster 2 (weekend-heavy) - shifting high-power tasks to weekends


In [5]:
def get_recommendations(cluster_id):
    recommendations = {
        0: [
            "🔥 Segment Archetype: Thermal/Heating Sensitive Home",
            "📊 Observation: Your energy usage spikes dramatically on cold winter days, suggesting electric heating.",
            "🛠️ Conservation Strategy: Inspect home insulation, loft seals, and window glazing. A smart thermostat (e.g. Nest, Hive) can save up to 15% on heating costs.",
            "🔌 Action: Set thermostatic radiator valves (TRVs) to heat occupied rooms only."
        ],
        1: [
            "💡 Segment Archetype: Frugal & Efficient Home",
            "📊 Observation: Your base energy usage is low and highly efficient.",
            "🛠️ Tariff Strategy: Switch to a ToU (Time-of-Use) tariff if you aren't on one. Shift laundry/dishwashing to late nights or early mornings to buy electricity at ultra-cheap off-peak rates.",
            "🔌 Action: Automate smart appliances to run before 16:00 or after 19:00."
        ],
        2: [
            "⚖️ Segment Archetype: Weekend Centric Consumption Profile",
            "📊 Observation: Your energy consumption is significantly higher on weekends than weekdays (weekend bias > 1.2).",
            "🛠️ Shifting Strategy: You are an excellent candidate for weekend-incentive tariffs. You can save substantially by shifting high-power tasks (e.g. washing, EV charging, deep cleaning) entirely to weekend daytime hours.",
            "🔌 Action: Use smart schedules to delay heavy appliance operations to Saturday/Sunday mornings and afternoons."
        ]
    }
    return recommendations.get(cluster_id, ["No recommendations available."])


## 5. Pipeline Class

Wrapping the forecaster, scaler, clusterer and recommendation logic into one class so it's easy to call for a given household.


In [10]:
class EnergyLensPipeline:
    def __init__(self, models_dir, profiles_path):
        self.forecaster = joblib.load(os.path.join(models_dir, 'xgboost_forecaster.joblib'))
        self.scaler = joblib.load(os.path.join(models_dir, 'scaler.joblib'))
        self.clusterer = joblib.load(os.path.join(models_dir, 'kmeans_clusterer.joblib'))
        self.profiles = pd.read_csv(profiles_path)

        self.forecast_features = [
    'energy_lag_1',
    'energy_lag_7',
    'energy_lag_14',
    'energy_lag_28',
    'energy_roll_mean_7',
    'energy_roll_std_7',
    'energy_roll_mean_30',
    'temp_avg',
    'HDD',
    'CDD',
    'temp_range',
    'is_weekend',
    'is_holiday',
    'tariff_code',
    'acorn_code'
]

    def _build_recursive_features(self, history_values, weather_row):
        s = pd.Series(history_values, dtype='float64')

        def lag(n):
            return float(s.iloc[-n]) if len(s) >= n else float(s.iloc[0])

        def roll_mean(n):
            return float(s.tail(min(n, len(s))).mean())

        def roll_std(n):
            value = float(s.tail(min(n, len(s))).std(ddof=0))
            return 0.0 if np.isnan(value) else value

        return {
               'energy_lag_1': lag(1),
                'energy_lag_7': lag(7),
                'energy_lag_14': lag(14),
                'energy_lag_28': lag(28),

                'energy_roll_mean_7': roll_mean(7),
                'energy_roll_std_7': roll_std(7),
                'energy_roll_mean_30': roll_mean(30),

                'temp_avg': float(weather_row['temp_avg']),
                'HDD': float(weather_row['HDD']),
                'CDD': float(weather_row['CDD']),
                'temp_range': float(weather_row['temp_range']),

                'is_weekend': int(weather_row['is_weekend']),
                'is_holiday': int(weather_row['is_holiday']),

                'tariff_code': int(weather_row['tariff_code']),
                'acorn_code': int(weather_row['acorn_code'])
        }

    def forecast_and_analyze(self, household_id, recent_history, weather_forecast):
        profile_row = self.profiles[self.profiles['LCLid'] == household_id]
        if profile_row.empty:
            raise ValueError(f'Household {household_id} not found in profiles.')

        cluster_id = int(profile_row['cluster'].iloc[0])
        archetypes = {
            0: 'Thermal/Heating Sensitive Home',
            1: 'Low-Usage Stable Consumption Profile',
            2: 'Weekend Centric Consumption Profile'
        }

        history = recent_history.sort_values('day')['energy_mean'].astype(float).tolist()
        if len(history) < 30:
            raise ValueError('At least 30 days of recent history are required.')

        forecasts = []
        for _, weather_row in weather_forecast.sort_values('day').iterrows():
            feature_dict = self._build_recursive_features(history, weather_row)
            X_future = pd.DataFrame([feature_dict], columns=self.forecast_features)
            prediction = float(self.forecaster.predict(X_future)[0])
            forecasts.append(round(prediction, 3))
            history.append(prediction)  # recursive update for the next forecast day

        recent_values = recent_history['energy_mean'].astype(float)
        mean_val = recent_values.mean()
        std_val = recent_values.std()
        if pd.isna(std_val) or std_val == 0:
            anomalous_dates = []
        else:
            z_scores = ((recent_values - mean_val) / std_val).abs()
            anomalous_dates = recent_history.loc[z_scores > 3, 'day'].dt.strftime('%Y-%m-%d').tolist()

        return {
            'household_id': str(household_id),
            'cluster_id': cluster_id,
            'cluster_archetype': archetypes.get(cluster_id, 'Unknown Profile'),
            '7_day_forecast': forecasts,
            'forecast_dates': pd.to_datetime(weather_forecast['day']).dt.strftime('%Y-%m-%d').tolist(),
            'anomalous_days_detected': anomalous_dates,
            'personalized_recommendations': get_recommendations(cluster_id)
        }

## 6. Testing the Pipeline

Running one sample household through the pipeline end-to-end to check everything works together.


In [7]:
# Verify serialized artifacts before inference
required_artifacts = [
    os.path.join(models_dir, 'xgboost_forecaster.joblib'),
    os.path.join(models_dir, 'scaler.joblib'),
    os.path.join(models_dir, 'kmeans_clusterer.joblib')
]

for path in required_artifacts:
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing model artifact: {path}')

print('✅ All model artifacts are ready for inference.')

✅ All model artifacts are ready for inference.


In [11]:
# Instantiate pipeline using the paths already created in this notebook
pipeline = EnergyLensPipeline(
    models_dir=models_dir,
    profiles_path=profiles_path
)

# Select a sample household
sample_hh = profiles['LCLid'].iloc[0]

# Use the latest 30 observed days for recursive forecasting
recent_history = (
    df[df['LCLid'] == sample_hh]
    .sort_values('day')
    .tail(30)
    .copy()
)

# Get household-specific categorical features
tariff_code = int(recent_history['tariff_code'].iloc[-1])
acorn_code = int(recent_history['acorn_code'].iloc[-1])

# Simulate a 7-day upcoming weather forecast
upcoming_dates = pd.date_range(
    start='2014-03-01',
    periods=7,
    freq='D'
)

weather_forecast = pd.DataFrame({
    'day': upcoming_dates,
    'temp_avg': [6.5, 5.0, 4.2, 5.5, 7.0, 8.2, 6.0],
    'HDD': [9.0, 10.5, 11.3, 10.0, 8.5, 7.3, 9.5],
    'CDD': [0.0] * 7,
    'temp_range': [4.0, 5.0, 3.5, 4.0, 5.5, 6.0, 4.5],
    'is_weekend': [1, 1, 0, 0, 0, 0, 0],
    'is_holiday': [0] * 7,

    # Household-specific features required by XGBoost
    'tariff_code': [tariff_code] * 7,
    'acorn_code': [acorn_code] * 7
})

# Run recursive 7-day forecast
output = pipeline.forecast_and_analyze(
    sample_hh,
    recent_history,
    weather_forecast
)

# Print final output
import json

print('\n✅ FINAL PIPELINE OUTPUT')
print(json.dumps(output, indent=2))


✅ FINAL PIPELINE OUTPUT
{
  "household_id": "MAC000002",
  "cluster_id": 2,
  "cluster_archetype": "Weekend Centric Consumption Profile",
  "7_day_forecast": [
    0.42,
    0.41,
    0.371,
    0.385,
    0.374,
    0.382,
    0.398
  ],
  "forecast_dates": [
    "2014-03-01",
    "2014-03-02",
    "2014-03-03",
    "2014-03-04",
    "2014-03-05",
    "2014-03-06",
    "2014-03-07"
  ],
  "anomalous_days_detected": [
    "2014-02-28"
  ],
  "personalized_recommendations": [
    "\u2696\ufe0f Segment Archetype: Weekend Centric Consumption Profile",
    "\ud83d\udcca Observation: Your energy consumption is significantly higher on weekends than weekdays (weekend bias > 1.2).",
    "\ud83d\udee0\ufe0f Shifting Strategy: You are an excellent candidate for weekend-incentive tariffs. You can save substantially by shifting high-power tasks (e.g. washing, EV charging, deep cleaning) entirely to weekend daytime hours.",
    "\ud83d\udd0c Action: Use smart schedules to delay heavy appliance ope

## 7. Summary

The final pipeline brings together:
1. A household-level XGBoost forecaster
2. Recursive 7-day forecasting, where each day's prediction feeds into the lag/rolling features for the next day
3. Household clustering using the saved scaler and KMeans model
4. Anomaly detection on recent history
5. Cluster-based recommendations
6. Saved model files that can be reloaded without retraining

That wraps up the EnergyLens-AI project - EDA, cleaning, feature engineering, model comparison, clustering/anomaly detection, and finally this combined pipeline.
